**13/08/2026** -- Inline results for section *Relationship between deprivation and fire-related PM2.5 exposure*


In [2]:
library(dplyr)
library(readr)
library(tidyr)
library(stringr)
library(brms)
library(rstan)
library(cmdstanr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Rcpp

Loading 'brms' package (version 2.22.0). Useful instructions
can be found by typing help('brms'). A more detailed introduction
to the package is available through vignette('brms_overview').


Attaching package: ‘brms’


The following object is masked from ‘package:stats’:

    ar


Loading required package: StanHeaders


rstan version 2.32.6 (Stan version 2.32.2)


For execution on a local, multicore CPU with excess RAM we recommend calling
options(mc.cores = parallel::detectCores()).
To avoid recompilation of unchanged Stan programs, we recommend calling
rstan_options(auto_write = TRUE)
For within-chain threading using `reduce_sum()` or `map_rect()` Stan functions,
change `threads_per_chain` option:
rstan_options(threads_per_chain = 1)



Attaching package

In [3]:
root <- rprojroot::find_root(rprojroot::has_file(".gitignore"))
source(file.path(root, "src/deprivation_pm_bhm/data_prep.R"))
source(file.path(root, "src/deprivation_pm_bhm/pca.R"))
source(file.path(root, "src/deprivation_pm_bhm/model_setup.R"))
source(file.path(root, "src/deprivation_pm_bhm/model_summary.R"))
source(file.path(root, "src/deprivation_pm_bhm/postprocess.R"))
source(file.path(root, "src/utils/utils.R"))

In [4]:
# Function to post-process model
postprocess <- function(model, df) {
    draws   <- as_draws_df(model)

    # PC1 effects in U & R (draws)
    pc1_draws_ur <- extract_pc1_draws_ur(draws)      
    
    # Posterior summary by country & U/R
    slopes_ur <- summarise_draws(
        pc1_draws_ur, 
        pivot = TRUE, pivot_cols = c("rural", "urban"),
        names_to = "urban_rural_cat",
        group_vars = c("country", "urban_rural_cat", "term")
    )
    
    # Post-stratification: p-w avg of U & R PC1 draws
    pc1_draws_pw_avg <- compute_pw_pc1_draws(pc1_draws_ur, df)
    
    # Posterior summary of U/R pooled effects draws
    slopes_pw_avg <- summarise_draws( 
        pc1_draws_pw_avg, 
        pivot = TRUE, pivot_cols = c("pooled"),
        names_to = "urban_rural_cat",
        group_vars = c("country", "urban_rural_cat", "term")
    )

    list(
        "slopes_ur"     = slopes_ur,
        "slopes_pw_avg" = slopes_pw_avg,
        "pc1_draws_ur"  = pc1_draws_ur,
        "pc1_draws_pw_avg" = pc1_draws_pw_avg
    )
}

#### Model for 2000-2022
With imputed socioeconomic data

In [5]:
# Load config
cfg             <- yaml::read_yaml(file.path(root, "configs/bhm_fit_ppca_hu_15kiter_24threads_cfg.yaml"))
data_filepath   <- cfg$project$data_filepath
model_save_dir  <- cfg$project$model_save_dir

In [6]:
set_cmdstan_path(cfg$environment$cmdstan_path)

CmdStan path set to: /home/users/cho00/miniconda3/envs/ppca/bin/cmdstan



In [7]:
# Construct model file path
model_file <- build_modelfile(
    out_path    = model_save_dir,
    model_name  = cfg$model$name,
    outcome     = cfg$model$outcome,
    scale_y     = cfg$data$scale_y,
    pca_method  = cfg$pca$method,
    n_pcs       = cfg$pca$n_pcs,    
    mcmc_cfg    = cfg$mcmc,
    priors_cfg  = cfg$priors,
    max_year    = cfg$data$max_year
)

In [8]:
# Load model
model <- readRDS(model_file)

In [9]:
summary(model)

 Family: gaussian 
  Links: mu = identity; sigma = identity 
Formula: fire_PM25_hu ~ 0 + Intercept + PC1 * urban_rural_cat + year + (1 + PC1 * urban_rural_cat + year | country) 
   Data: data_mod (Number of observations: 920983) 
  Draws: 4 chains, each with iter = 15000; warmup = 5000; thin = 1;
         total post-warmup draws = 40000

Multilevel Hyperparameters:
~country (Number of levels: 52) 
                                                   Estimate Est.Error l-95% CI
sd(Intercept)                                          3.08      0.30     2.55
sd(PC1)                                                1.37      0.13     1.14
sd(urban_rural_caturban)                               0.84      0.09     0.67
sd(year)                                               0.07      0.01     0.06
sd(PC1:urban_rural_caturban)                           0.98      0.11     0.79
cor(Intercept,PC1)                                    -0.02      0.13    -0.26
cor(Intercept,urban_rural_caturban)           

In [18]:
# Load data
df <- readr::read_csv(data_filepath) |>
    filter(year <= cfg$data$max_year) |>
    group_by(lon, lat) |>
    mutate(grid_id = cur_group_id()) |> # create grid_id for projecting SE vars
    ungroup()

Rows: 994412 Columns: 74
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (63): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [19]:
# Impute missing 2018-2022 socioeconomic indicators
df <- project_indicators(df, cfg$data$indicator_projections)

[1] "Projecting edu_mean_years forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0227167 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


[1] "Projecting imp_san_access_pct forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.00349425 (tol = 0.002, component 1)”


[1] "Projecting stunting_pct_u5 forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0278442 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


In [22]:
# Do PCA
pca_res <- compute_pca(
    df,
    method          = cfg$pca$method,
    se_indicators   = cfg$data$se_indicators,
    n_pcs           = cfg$pca$n_pcs,
    scale           = cfg$pca$scale,
    centre          = cfg$pca$centre,
    seed            = cfg$pca$seed,
    positive_vars   = cfg$pca$pc1_positive_loadings,
    negative_vars   = cfg$pca$pc1_negative_loadings
)

[2026-08-13 18:07:20] Flipping PC1 so higher = more deprivation



In [24]:
# Join PCs back onto df (helper funct)
df  <- prepare_analysis_data(df, 
                             pca_res$data, 
                             outcome = "fire_PM25_hu", 
                             scale_y = cfg$data$scale_y,
                             scale_PC1 = cfg$data$scale_PC1)

In [34]:
slopes <- postprocess(model, df)

Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”


##### “In rural settings, associations ranged from X to Y µg/m3 per SD deprivation; in urban settings, from X to Y µg/m3”

In [42]:
slopes$slopes_ur |> 
    group_by(urban_rural_cat) |>
    filter(
        estimate == min(estimate) |
        estimate == max(estimate)
    ) |> 
    arrange(urban_rural_cat, estimate)

country,urban_rural_cat,term,estimate,conf.low,conf.high,median
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
South Sudan,rural,PC1,-4.188656,-4.280200,-4.098163,-4.188549
"Congo, Rep. of",rural,PC1,3.564896,3.386854,3.745250,3.564077
Nigeria,urban,PC1,-1.364307,-1.478403,-1.248625,-1.364442
"Congo, Rep. of",urban,PC1,2.190327,1.585736,2.785532,2.192660


##### “Continental pooled estimates were near zero (𝛽rural = X µg/m3 (95% CrI: Y to Z); 𝛽urban = X µg/m3 (Y to Z)), reflecting the diversity of positive and negative country-specific associations”

In [43]:
slopes$slopes_ur |> 
    filter(country == "Pooled, all countries")

country,urban_rural_cat,term,estimate,conf.low,conf.high,median
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
"Pooled, all countries",rural,PC1,0.1053254,-0.2813821,0.4832029,0.1073315
"Pooled, all countries",urban,PC1,0.4005338,0.1241196,0.6696961,0.4020803


##### “Within rural contexts, more deprived areas had higher fire PM2.5 in X of 52 countries and lower exposure in Y countries; in the remaining Z countries, associations were not distinguishable from zero with posterior probability (PP) >0.975”

In [46]:
slopes$slopes_ur |> 
    filter(country != "Pooled, all countries") |>
    filter(urban_rural_cat == "rural") |>
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,18
null,11
positive,23


##### “Higher exposure among more deprived urban areas was observed in X countries, lower exposure in Y countries, and no detectable association was observed in Z countries with PP >0.975.”

In [47]:
slopes$slopes_ur |> 
    filter(country != "Pooled, all countries") |>
    filter(urban_rural_cat == "urban") |>
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,3
null,32
positive,17


#### Model for 2000-2017
Without imputed socioeconomic data

In [48]:
# Load config
cfg             <- yaml::read_yaml(file.path(root, "configs/bhm_fit_ppca_hu_2000_2017_15kiter_24threads_cfg.yaml"))
data_filepath   <- cfg$project$data_filepath
model_save_dir  <- cfg$project$model_save_dir

In [50]:
set_cmdstan_path(cfg$environment$cmdstan_path)

CmdStan path set to: /home/users/cho00/miniconda3/envs/ppca/bin/cmdstan



In [51]:
# Construct model file path
model_file <- build_modelfile(
    out_path    = model_save_dir,
    model_name  = cfg$model$name,
    outcome     = cfg$model$outcome,
    scale_y     = cfg$data$scale_y,
    pca_method  = cfg$pca$method,
    n_pcs       = cfg$pca$n_pcs,    
    mcmc_cfg    = cfg$mcmc,
    priors_cfg  = cfg$priors,
    max_year    = cfg$data$max_year
)

In [52]:
# Load model
model <- readRDS(model_file)

In [53]:
# Load data
df <- readr::read_csv(data_filepath) |>
    filter(year <= cfg$data$max_year) |>
    group_by(lon, lat) |>
    mutate(grid_id = cur_group_id()) |> # create grid_id for projecting SE vars (note not needed here)
    ungroup()

Rows: 994412 Columns: 74
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (63): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [56]:
# Do PCA
pca_res <- compute_pca(
    df,
    method          = cfg$pca$method,
    se_indicators   = cfg$data$se_indicators,
    n_pcs           = cfg$pca$n_pcs,
    scale           = cfg$pca$scale,
    centre          = cfg$pca$centre,
    seed            = cfg$pca$seed,
    positive_vars   = cfg$pca$pc1_positive_loadings,
    negative_vars   = cfg$pca$pc1_negative_loadings
)

In [57]:
# Join PCs back onto df (helper funct)
df  <- prepare_analysis_data(df, 
                             pca_res$data, 
                             outcome = "fire_PM25_hu", 
                             scale_y = cfg$data$scale_y,
                             scale_PC1 = cfg$data$scale_PC1)

In [58]:
slopes <- postprocess(model, df)

Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”


##### “In rural settings, associations ranged from X to Y µg/m3 per SD deprivation; in urban settings, from X to Y µg/m3”

In [59]:
slopes$slopes_ur |> 
    group_by(urban_rural_cat) |>
    filter(
        estimate == min(estimate) |
        estimate == max(estimate)
    ) |> 
    arrange(urban_rural_cat, estimate)

country,urban_rural_cat,term,estimate,conf.low,conf.high,median
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
South Sudan,rural,PC1,-4.613980,-4.710219,-4.517836,-4.614124
"Congo, Rep. of",rural,PC1,3.656958,3.449337,3.863450,3.657453
Chad,urban,PC1,-3.676564,-6.277653,-1.171872,-3.668503
Zambia,urban,PC1,2.281300,1.411082,3.169929,2.280337


##### “Continental pooled estimates were near zero (𝛽rural = X µg/m3 (95% CrI: Y to Z); 𝛽urban = X µg/m3 (Y to Z)), reflecting the diversity of positive and negative country-specific associations”

In [60]:
slopes$slopes_ur |> 
    filter(country == "Pooled, all countries")

country,urban_rural_cat,term,estimate,conf.low,conf.high,median
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
"Pooled, all countries",rural,PC1,0.12146921,-0.3154426,0.5410465,0.12301996
"Pooled, all countries",urban,PC1,0.07342696,-0.3543934,0.4691492,0.07846668


##### “Within rural contexts, more deprived areas had higher fire PM2.5 in X of 52 countries and lower exposure in Y countries; in the remaining Z countries, associations were not distinguishable from zero with posterior probability (PP) >0.975”

In [61]:
slopes$slopes_ur |> 
    filter(country != "Pooled, all countries") |>
    filter(urban_rural_cat == "rural") |>
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,16
null,13
positive,23


##### “Higher exposure among more deprived urban areas was observed in X countries, lower exposure in Y countries, and no detectable association was observed in Z countries with PP >0.975.”

In [62]:
slopes$slopes_ur |> 
    filter(country != "Pooled, all countries") |>
    filter(urban_rural_cat == "urban") |>
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,7
null,34
positive,11
